In [87]:
import numpy as np
from dotenv import load_dotenv
import pandas as pd
import os
import pandas as pd
from pathlib import Path
from IPython.display import display
from locale import atof
from openpyxl.styles import numbers
from datetime import datetime

BASE_DIR = Path('/Users/andresuchitra/dev/missglam/autopo/notebook')


# Configuration
SUPPLIER_PATH = BASE_DIR / 'data/supplier.csv'
RAWPO_DIR = BASE_DIR / 'data/rawpo/csv'
INPUT_DIR = BASE_DIR / 'data/input'
RAWPO_XLSX_DIR = BASE_DIR / 'data/rawpo/xlsx'
STORE_CONTRIBUTION_PATH = BASE_DIR / 'data/store_contribution.csv'
OUTPUT_DIR = BASE_DIR / 'output/complete'
OUTPUT_EXCEL_DIR = BASE_DIR / 'output/excel'
OUTPUT_M2_DIR = BASE_DIR / 'output/m2'
OUTPUT_EMERGENCY_DIR = BASE_DIR / 'output/emergency'

TOP_100_SKU_DIR = BASE_DIR / 'data/top_100_sku'

NUMERIC_COLUMNS = [
    'HPP', 'Harga', 'Ranking', 'Grade', 'Terjual', 'Stok', 'Lost Days',
    'Velocity Capped', 'Daily Sales', 'Lead Time', 'Max. Daily Sales',
    'Max. Lead Time', 'Min. Order', 'Safety Stok', 'ROP', '3W Cover',
    'Sedang PO', 'Suggested', 'Amount', 'Promo Factor', 'Delay Factor',
    'Stock Cover', 'Days to Backup', 'Qty to Backup'
]

NA_VALUES = {
    'NAN', 'NA', '#N/A', 'NULL', 'NONE', '', '?', '-', 'INF', '-INF',
    '+INF', 'INFINITY', '-INFINITY', '1.#INF', '-1.#INF', '1.#QNAN'
}

def format_number_for_csv(x):
    """Format numbers for CSV output with Indonesian locale (comma as decimal, dot as thousand)"""
    if pd.isna(x) or x == '':
        return x
    try:
        if isinstance(x, (int, float)):
            if x == int(x):  # Whole number
                return f"{int(x):,d}".replace(",", ".")
            else:  # Decimal number
                return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        return x
    except:
        return x

def get_store_name_from_filename(filename):
    """Extract store name from filename, handling different patterns."""
    # Remove file extension and split by spaces
    name_parts = Path(filename).stem.split()
    
    # Handle cases like "002 Miss Glam Pekanbaru.csv" -> "Pekanbaru"
    # or "01 Miss Glam Padang.csv" -> "Padang"
    if len(name_parts) >= 3 and name_parts[1].lower() == 'miss' and name_parts[2].lower() == 'glam':
        return ' '.join(name_parts[3:]).strip().upper()
    elif len(name_parts) >= 2 and name_parts[0].lower() == 'miss' and name_parts[1].lower() == 'glam':
        return ' '.join(name_parts[2:]).strip().upper()
    # Fallback: take everything after the first space
    elif ' ' in filename:
        return ' '.join(name_parts[1:]).strip().upper()
    return name_parts[0].upper()

def read_csv_file(file_path):
    # List of (separator, encoding) combinations to try
    formats_to_try = [
        (',', 'utf-8'),      # Standard CSV with comma
        (';', 'utf-8'),      # Semicolon with UTF-8
        (',', 'latin1'),     # Comma with Latin1
        (';', 'latin1'),     # Semicolon with Latin1
        (',', 'cp1252'),     # Windows-1252 encoding
        (';', 'cp1252')
    ]
    
    for sep, enc in formats_to_try:
        try:
            df = pd.read_csv(
                file_path,
                sep=sep,
                decimal=',',
                thousands='.',
                encoding=enc,
                engine='python'  # More consistent behavior with Python engine
            )
            # If we get here, the file was read successfully
            if not df.empty:
                return df
        except (UnicodeDecodeError, pd.errors.ParserError, pd.errors.EmptyDataError) as e:
            continue  # Try next format
        except Exception as e:
            print(f"Unexpected error reading {file_path} with sep='{sep}', encoding='{enc}': {str(e)}")
            continue
    
    # If we get here, all attempts failed
    print(f"Failed to read {file_path} with any known format")
    return None

def read_csv_file_v2(file_path, decimal_sep=',', thousand_sep=None):
    # List of (separator, encoding) combinations to try
    formats_to_try = [
        (',', 'utf-8'),      # Standard CSV with comma
        (';', 'utf-8'),      # Semicolon with UTF-8
        (',', 'latin1'),     # Comma with Latin1
        (';', 'latin1'),     # Semicolon with Latin1
        (',', 'cp1252'),     # Windows-1252 encoding
        (';', 'cp1252')
    ]
    
    for sep, enc in formats_to_try:
        try:
            df = pd.read_csv(
                file_path,
                sep=sep,
                decimal=decimal_sep,
                thousands=thousand_sep,
                encoding=enc,
                engine='python'  # More consistent behavior with Python engine
            )
            # If we get here, the file was read successfully
            if not df.empty:
                return df
        except (UnicodeDecodeError, pd.errors.ParserError, pd.errors.EmptyDataError) as e:
            continue  # Try next format
        except Exception as e:
            print(f"Unexpected error reading {file_path} with sep='{sep}', encoding='{enc}': {str(e)}")
            continue
    
    # If we get here, all attempts failed
    print(f"Failed to read {file_path} with any known format")
    return None

# SAVE AND READ functions

# Apply the formatting to numeric columns in your final output
def format_dataframe_display(df):
    # Make a copy to avoid SettingWithCopyWarning
    df_display = df.copy()
    
    # Apply formatting to numeric columns
    for col in df_display.select_dtypes(include=['int64', 'float64']).columns:
        df_display[col] = df_display[col].apply(
            lambda x: format_id_number(x, 2) if pd.notna(x) else x
        )
    
    return df_display

def read_excel_file(file_path):
    """
    Read an Excel file with robust error handling for problematic values.
    """
    try:
        print(f"\nReading excel file: {file_path.name}...")
        
        # First, read the file with openpyxl directly to handle the data more carefully
        from openpyxl import load_workbook
        
        # Load the workbook
        wb = load_workbook(
            filename=file_path,
            read_only=True,    # Read-only mode is faster and uses less memory
            data_only=True,    # Get the stored value instead of the formula
            keep_links=False   # Don't load external links
        )
        
        # Get the first sheet
        ws = wb.active
        
        # Get headers from the first row
        headers = []
        for idx, cell in enumerate(next(ws.iter_rows(values_only=True))):
            header = str(cell).strip() if cell not in (None, '') else f"Column_{idx + 1}"
            headers.append(header)
        
        # Initialize data rows
        data = []
        
        # Process each row
        for row in ws.iter_rows(min_row=2, values_only=True):  # Skip header row
            row_data = []
            for cell in row:
                if cell is None:
                    row_data.append('')
                    continue

                cell_str = str(cell).strip()
                if cell_str.upper() in NA_VALUES:
                    row_data.append('')
                else:
                    row_data.append(cell_str)
            
            # Only add row if it has data
            if any(cell != '' for cell in row_data):
                data.append(row_data)
        
        # Create DataFrame
        df = pd.DataFrame(data, columns=headers)
        
        # Normalize column data types
        df = clean_and_convert(df)
        
        print(f"✅ Successfully processed {file_path.name} with {len(df)} rows")
        return df
        
    except Exception as e:
        print(f"❌ Error processing {file_path.name}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

def save_file(df, file_path, file_format='csv', **kwargs):
    """
    Save DataFrame to file with consistent extension and content type.
    
    Args:
        df: DataFrame to save
        file_path: Path object or string for the output file
        file_format: 'csv' or 'xlsx'
        **kwargs: Additional arguments to pass to to_csv or to_excel
        
    Returns:
        Path: The path where the file was saved
    """
    # Ensure file_path is a Path object
    file_path = Path(file_path)
    
    # Ensure the directory exists
    file_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Ensure the correct file extension
    if not file_path.suffix.lower() == f'.{file_format}':
        file_path = file_path.with_suffix(f'.{file_format}')
    
    # Make a copy to avoid modifying the original
    df_output = df.copy()
    
    # Common preprocessing
    if 'SKU' in df_output.columns:
        df_output['SKU'] = df_output['SKU'].astype(str).str.strip()
        if file_format == 'xlsx':
            # For Excel, wrap SKU in ="..." to preserve leading zeros
            df_output['SKU'] = df_output['SKU'].apply(lambda x: f'="{x}"')
    
    # Format numbers for CSV if needed
    if file_format == 'csv':
        numeric_cols = df_output.select_dtypes(include=['number']).columns
        for col in numeric_cols:
            df_output[col] = df_output[col].apply(format_number_for_csv)
    
    # Save based on format
    if file_format == 'csv':
        df_output.to_csv(
            file_path, 
            index=False, 
            sep=';', 
            decimal=',', 
            encoding='utf-8-sig',
            **kwargs
        )
    elif file_format == 'xlsx':
        with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
            df_output.to_excel(writer, index=False, **kwargs)
            
            # Format SKU column as text in Excel
            if 'SKU' in df_output.columns:
                ws = writer.sheets[list(writer.sheets.keys())[0]]
                sku_col_idx = df_output.columns.get_loc("SKU") + 1
                for row in ws.iter_rows(
                    min_row=2,  # Skip header
                    max_row=ws.max_row,
                    min_col=sku_col_idx,
                    max_col=sku_col_idx
                ):
                    for cell in row:
                        cell.number_format = numbers.FORMAT_TEXT
    else:
        raise ValueError(f"Unsupported file format: {file_format}")
    
    print(f"File saved to {file_path}")
    return file_path

def save_to_complete_format(df, filename, file_format='csv', **kwargs):
    """
    Save Complete format file with consistent extension.
    
    Args:
        df: Input DataFrame
        filename: Output filename (with or without extension)
        file_format: 'csv' or 'xlsx'
        **kwargs: Additional arguments for save_file
    """
    
    # Ensure output directory exists
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # Save with consistent extension
    output_path = OUTPUT_DIR / filename

    return save_file(df, output_path, file_format=file_format, **kwargs)

def save_to_m2_format(df, filename, file_format='csv', **kwargs):
    """
    Save M2 format file with consistent extension.
    
    Args:
        df: Input DataFrame
        filename: Output filename (with or without extension)
        file_format: 'csv' or 'xlsx'
        **kwargs: Additional arguments for save_file
    """
    # Filter to only include rows with regular PO qty > 0
    df_filtered = df[df['final_updated_regular_po_qty'] > 0].copy()
    df_output = df_filtered[['Toko', 'Brand', 'SKU', 'HPP', 'final_updated_regular_po_qty']]
    
    # Ensure output directory exists
    OUTPUT_M2_DIR.mkdir(parents=True, exist_ok=True)
    
    # Save with consistent extension
    output_path = OUTPUT_M2_DIR / filename
    return save_file(df_output, output_path, file_format=file_format, **kwargs)

def save_to_emergency_po_format(df, filename, file_format='csv', **kwargs):
    """
    Save emergency PO format file with consistent extension.
    
    Args:
        df: Input DataFrame
        filename: Output filename (with or without extension)
        file_format: 'csv' or 'xlsx'
        **kwargs: Additional arguments for save_file
    """
    # Filter to only include rows with emergency PO qty > 0
    df_filtered = df[df['emergency_po_qty'] > 0].copy()
    df_output = df_filtered[[
        'Brand', 'SKU', 'Nama', 'Toko', 'HPP', 
        'emergency_po_qty', 'emergency_po_cost'
    ]]

    # Ensure output directory exists
    OUTPUT_EMERGENCY_DIR.mkdir(parents=True, exist_ok=True)
    
    # Save with consistent extension
    output_path = OUTPUT_EMERGENCY_DIR / filename
    return save_file(df_output, output_path, file_format=file_format, **kwargs)
    

In [88]:
# Formatter
def normalize_date_string(date_str):
    """Convert Indonesian month names to English for consistent parsing."""

    # Month mapping: Indonesian -> English
    month_mapping = {
        'jan': 'jan', 'januari': 'january',
        'feb': 'feb', 'februari': 'february',
        'mar': 'mar', 'maret': 'march',
        'apr': 'apr', 'april': 'april',
        'mei': 'may', 'may': 'may',
        'jun': 'jun', 'juni': 'june',
        'jul': 'jul', 'juli': 'july',
        'agu': 'aug', 'agustus': 'august',
        'sep': 'sep', 'september': 'september',
        'okt': 'oct', 'oktober': 'october',
        'nov': 'nov', 'november': 'november',
        'des': 'dec', 'desember': 'december',
        'dec': 'dec'
    }

    date_lower = str(date_str).lower().strip()
    for indo_month, eng_month in month_mapping.items():
        # Use word boundaries to avoid partial matches
        import re
        date_lower = re.sub(r'\b' + indo_month + r'\b', eng_month, date_lower)

    return date_lower

def getFrameSpecificDate(df: DataFrame, target_date: str):
    """
    Match date columns in DataFrame with target_date, supporting both English and Indonesian month names.
    
    Args:
        df: DataFrame with date columns
        target_date: Target date string (e.g., "18 Dec 2025" or "18 Des 2025")
    
    Returns:
        tuple: (matched_columns, filtered_dataframe)
    """
    if df is None or df.empty:
        return [], df
    
    # Normalize and parse target date
    normalized_target = normalize_date_string(target_date)
    target_ts = pd.to_datetime(normalized_target, dayfirst=True, errors='coerce')

    if pd.isna(target_ts):
        raise ValueError(f"Unable to parse target_date: {target_date} (normalized: {normalized_target})")

    has_year = bool(pd.Series([target_date]).str.contains(r"\b\d{4}\b", regex=True).iloc[0])
    matched_cols = []

    for col in df.columns:
        col_str = str(col).strip()
        normalized_col = normalize_date_string(col_str)
        col_ts = pd.to_datetime(normalized_col, dayfirst=True, errors='coerce')
        if pd.isna(col_ts):
            continue
        if has_year:
            if col_ts.normalize() == target_ts.normalize():
                matched_cols.append(col)
        else:
            if (col_ts.day == target_ts.day) and (col_ts.month == target_ts.month):
                matched_cols.append(col)
    return matched_cols, df.loc[:, matched_cols] if matched_cols else df.iloc[:, 0:0]

In [89]:
# Load Top 100 SKU from specific DIR and Top 100 Special Handlings

def load_top_100_sku_for_store(location, top_100_dir=TOP_100_SKU_DIR, expected_header_keys=None, max_rows=100):
    """Load Top 100 SKU data for a given store.

    This function will:
    - Find the matching Top 100 file for the store in ``top_100_dir``
    - Detect which row actually contains the header (can be on row 1, 2, 3, ...)
    - Read and return the cleaned DataFrame limited to top ``max_rows`` SKUs
    """
    try:
        location_upper = str(location).strip().upper()
        top_100_dir = Path(top_100_dir)

        if not top_100_dir.exists():
            print(f"Top 100 SKU directory does not exist: {top_100_dir}")
            return None

        # Use the same filename → store name logic to match files
        matching_files = []
        for path in sorted(top_100_dir.glob('*')):
            if not path.is_file():
                continue
            store_name = get_store_name_from_filename(path.name)

            if store_name.upper() == location_upper:
                matching_files.append(path)

        if not matching_files:
            print(f"No Top 100 SKU file found for store: {location_upper}")
            return None

        # If multiple files match, take the first one deterministically
        file_path = matching_files[0]
        print(f"Loading Top 100 SKU for {location_upper} from: {file_path}")

        suffix = file_path.suffix.lower()

        # Step 1: read raw file without assuming header row
        if suffix in ['.xlsx', '.xls']:
            raw_df = pd.read_excel(file_path, header=None, engine='openpyxl')
        else:
            read_ok = False
            raw_df = None
            for sep in [';', ',']:
                for enc in ['utf-8-sig', 'utf-8', 'latin1', 'cp1252']:
                    try:
                        raw_df = pd.read_csv(file_path, header=None, sep=sep, encoding=enc)
                        read_ok = True
                        break
                    except Exception:
                        continue
                if read_ok:
                    break

            if not read_ok or raw_df is None:
                print(f"Failed to read Top 100 SKU file: {file_path}")
                return None

        if raw_df is None or raw_df.empty:
            print(f"Top 100 SKU file is empty: {file_path}")
            return None

                # Step 2: detect which row is the header
        # Strategy:
        #   1. Prefer any row containing a cell equal to "sku" (case-insensitive)
        #   2. Otherwise, take the first non-empty row
        header_row = None
        max_header_search = min(15, len(raw_df))  # look a bit deeper if needed

        for idx in range(max_header_search):
            row_values = raw_df.iloc[idx].astype(str).str.strip()
            lowered = [v.lower() for v in row_values]

            if "sku" in lowered:
                header_row = idx
                break

        if header_row is None:
            # fallback: first non-empty row
            for idx in range(max_header_search):
                if raw_df.iloc[idx].notna().any():
                    header_row = idx
                    break

        # As a final fallback, if still None but we have rows, use row 0
        if header_row is None:
            header_row = 0

        # Step 3: build df from raw_df using that header row
        header = raw_df.iloc[header_row].astype(str).str.strip().tolist()
        data = raw_df.iloc[header_row + 1 :].reset_index(drop=True)
        data.columns = header
        df = data

        # Basic cleaning
        df.columns = df.columns.astype(str).str.strip()

        # Normalize SKU column if present under any casing
        sku_col = None
        for c in df.columns:
            if str(c).strip().lower() == "sku":
                sku_col = c
                break

        if sku_col is not None:
            df[sku_col] = df[sku_col].astype(str).str.strip()
            # also expose as "SKU" for downstream code
            if sku_col != "SKU":
                df["SKU"] = df[sku_col]

        # Drop completely empty rows
        df = df.dropna(how='all')

        # Limit to top N rows
        if max_rows is not None and max_rows > 0:
            df = df.head(max_rows)

        print(f"Loaded Top 100 SKU for {location_upper}: {len(df)} rows")
        return df

    except Exception as e:
        print(f"Error loading Top 100 SKU for {location}: {str(e)}")
        return None

def addTop100Stock(running_date: str):
    try:
        matched_cols, dec18_df = getFrameSpecificDate(df, "18 Dec 2025")
        if not matched_cols:
            print("No header matched the given date.")
        else:
            print(f"Matched columns: {matched_cols}")
            display(dec18_df)

    except Exception as e:
        print(f"Error: {e}. Column for the date may not be existing!")


In [90]:
def _patch_openpyxl_number_casting():
    """Ensure openpyxl won't crash when encountering NAN/INF in numeric cells."""
    print("Calling _patch_openpyxl_number_casting...")

    try:
        from openpyxl.worksheet import _reader

        original_cast = _reader._cast_number

        def _safe_cast_number(value):  # pragma: no cover - monkey patch
            if isinstance(value, str):
                if value.strip().upper() in NA_VALUES:
                    return 0
            try:
                return original_cast(value)
            except (ValueError, TypeError):
                return 0 if value in (None, '') else value

        _reader._cast_number = _safe_cast_number
    except Exception:
        # If patch fails we continue; runtime reader will still attempt default behaviour
        pass

In [91]:
# LOAD Supplier data, and Override merge_with_suppliers to prevent filling supplier data for rows with empty Brand
print("Applying empty-Brand-safe merge_with_suppliers override...")


def load_supplier_data(supplier_path):
    """Load and clean supplier data."""
    print(f"Loading supplier data: {supplier_path}")
    df = pd.read_csv(supplier_path, sep=';', decimal=',').fillna('')
    df['Nama Brand'] = df['Nama Brand'].str.strip()
    return df

def merge_with_suppliers_v1(df_clean, supplier_df):
    """Merge PO data with supplier information."""
    print("Merging with suppliers...")
    
    # Clean supplier data
    supplier_clean = supplier_df.copy()
    supplier_clean['Nama Brand'] = supplier_clean['Nama Brand'].astype(str).str.strip()
    supplier_clean['Nama Store'] = supplier_clean['Nama Store'].astype(str).str.strip()
    
    # Deduplicate to prevent row explosion - Unique Brand+Store
    supplier_clean = supplier_clean.drop_duplicates(subset=['Nama Brand', 'Nama Store'])
    
    # Ensure PO data has clean columns for merging
    df_clean['Brand'] = df_clean['Brand'].astype(str).str.strip()
    df_clean['Toko'] = df_clean['Toko'].astype(str).str.strip()
    
    # 1. Primary Merge: Match on Brand AND Store (Toko)
    # This prioritizes the specific supplier for that store
    merged_df = pd.merge(
        df_clean,
        supplier_clean,
        left_on=['Brand', 'Toko'],
        right_on=['Nama Brand', 'Nama Store'],
        how='left',
        suffixes=('_clean', '_supplier')
    )
    
    # 2. Fallback: For unmatched rows, try to find ANY supplier for that Brand
    # Identify rows where merge failed (Nama Brand is NaN)
    unmatched_mask = merged_df['Nama Brand'].isna()
    
    if unmatched_mask.any():
        print(f"Found {unmatched_mask.sum()} rows without direct store match. Attempting fallback...")
        
        # Get the unmatched rows and drop the empty supplier columns
        unmatched_rows = merged_df[unmatched_mask].copy()
        supplier_cols = [col for col in supplier_clean.columns if col in unmatched_rows.columns and col != 'Brand']
        unmatched_rows = unmatched_rows.drop(columns=supplier_cols)
        
        # Create fallback supplier list (one per brand)
        # We take the first one found for each brand
        fallback_suppliers = supplier_clean.drop_duplicates(subset=['Nama Brand'])
        
        # Merge unmatched rows with fallback suppliers
        matched_fallback = pd.merge(
            unmatched_rows,
            fallback_suppliers,
            left_on='Brand',
            right_on='Nama Brand',
            how='left',
            suffixes=('_clean', '_supplier')
        )
        
        # Combine the initially matched rows with the fallback-matched rows
        matched_initial = merged_df[~unmatched_mask]
        merged_df = pd.concat([matched_initial, matched_fallback], ignore_index=True)
    
    # Clean up supplier columns
    supplier_columns = [
        'ID Supplier', 'Nama Supplier', 'ID Brand', 'ID Store', 
        'Nama Store', 'Hari Order', 'Min. Purchase', 'Trading Term',
        'Promo Factor', 'Delay Factor'
    ]
    for col in supplier_columns:
        if col in merged_df.columns:
            merged_df[col] = merged_df[col].fillna('' if merged_df[col].dtype == 'object' else 0)
    
    return merged_df

def merge_with_suppliers(df_clean, supplier_df):
    """Merge PO data with supplier info, skipping fallback for blank Brand rows."""
    print("Merging with suppliers (override)...")

    supplier_clean = supplier_df.copy()
    supplier_clean['Nama Brand'] = supplier_clean['Nama Brand'].astype(str).str.strip()
    supplier_clean['Nama Store'] = supplier_clean['Nama Store'].astype(str).str.strip()
    supplier_clean = supplier_clean.drop_duplicates(subset=['Nama Brand', 'Nama Store'])

    df_clean = df_clean.copy()
    df_clean['Brand'] = df_clean['Brand'].astype(str).str.strip()
    df_clean['Toko'] = df_clean['Toko'].astype(str).str.strip()

    merged_df = pd.merge(
        df_clean,
        supplier_clean,
        left_on=['Brand', 'Toko'],
        right_on=['Nama Brand', 'Nama Store'],
        how='left',
        suffixes=('_clean', '_supplier')
    )

    merged_df['_brand_clean'] = merged_df['Brand'].astype(str).str.strip()
    unmatched_mask = merged_df['Nama Brand'].isna()
    fallback_mask = unmatched_mask & (merged_df['_brand_clean'] != '')

    if fallback_mask.any():
        print(
            f"Found {fallback_mask.sum()} rows without store match; attempting brand-only fallback for non-empty brands..."
        )
        unmatched_rows = merged_df[fallback_mask].copy()
        supplier_cols = [
            col for col in supplier_clean.columns if col in unmatched_rows.columns and col != 'Brand'
        ]
        unmatched_rows = unmatched_rows.drop(columns=supplier_cols, errors='ignore')
        unmatched_rows['Brand'] = unmatched_rows['_brand_clean']

        fallback_suppliers = supplier_clean.drop_duplicates(subset=['Nama Brand'])
        matched_fallback = pd.merge(
            unmatched_rows,
            fallback_suppliers,
            left_on='Brand',
            right_on='Nama Brand',
            how='left',
            suffixes=('_clean', '_supplier')
        )

        matched_initial = merged_df[~fallback_mask]
        merged_df = pd.concat([matched_initial, matched_fallback], ignore_index=True)

    merged_df = merged_df.drop(columns=['_brand_clean'], errors='ignore')

    supplier_columns = [
        'ID Supplier', 'Nama Supplier', 'ID Brand', 'ID Store',
        'Nama Store', 'Hari Order', 'Min. Purchase', 'Trading Term',
        'Promo Factor', 'Delay Factor'
    ]

    for col in supplier_columns:
        if col in merged_df.columns:
            merged_df[col] = merged_df[col].fillna('' if merged_df[col].dtype == 'object' else 0)

    return merged_df

Applying empty-Brand-safe merge_with_suppliers override...


In [ ]:
# main entry point execution

_patch_openpyxl_number_casting()

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_EXCEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_M2_DIR, exist_ok=True)
os.makedirs(OUTPUT_EMERGENCY_DIR, exist_ok=True)

EXCLUDE_BRANDS = ['WARDAH', 'MAKE OVER','OMG','EMINA','LABORE','KAHF','INSTAPERFECT','PUTRI','EARTH LOVE LIFE','TAVI','CRYSTALLURE','BIODEF']

STORES_18_DEC = [
    'Padang', 'Pekanbaru', 'Jambi', 'Bukittinggi', 'Muaro Bungo', 'Bengkulu', 'Medan', 'Palembang', 'Damar', 'Payakumbuh', 'Solok',
    'Lubuk Linggau', 'Kedaton', 'Dumai', 'Rantau Prapat', 'Tanjung Pinang', 'Sutomo', 'Pasaman Barat', 'Halat', 'Aceh', 'Mayang', 'Soeta'
]
STORES_17_DEC = ['Lampung', 'Bangka', 'Tembilahan', 'Duri', 'P. Sidimpuan', 'Sei Penuh']


def _build_stock_top100_map(df_top_100, running_date):
    """Build SKU+Toko -> stock_top100 mapping from date-specific column."""
    if df_top_100 is None or df_top_100.empty:
        print("  [DEBUG] _build_stock_top100_map: df_top_100 is None or empty")
        return None

    if 'SKU' not in df_top_100.columns:
        print("  [DEBUG] _build_stock_top100_map: 'SKU' column not found in df_top_100")
        return None

    if 'Toko' not in df_top_100.columns:
        print("  [DEBUG] _build_stock_top100_map: 'Toko' column not found in df_top_100")
        return None

    if not running_date:
        print("  [DEBUG] _build_stock_top100_map: running_date is None or empty")
        return None

    matched_cols, _ = getFrameSpecificDate(df_top_100, running_date)
    if not matched_cols:
        print(f"  [DEBUG] _build_stock_top100_map: No date column matched for '{running_date}'")
        print(f"  [DEBUG] Available columns: {df_top_100.columns.tolist()}")
        return None

    stock_date_col = matched_cols[0]
    print(f"  [DEBUG] _build_stock_top100_map: Using date column '{stock_date_col}' for running_date '{running_date}'")

    top = df_top_100[['SKU', 'Toko', stock_date_col]].copy()
    top['SKU'] = top['SKU'].astype(str).str.strip()
    top['Toko'] = top['Toko'].astype(str).str.strip()
    top['stock_top100'] = pd.to_numeric(top[stock_date_col], errors='coerce')
    top = top[['SKU', 'Toko', 'stock_top100']]
    top = top.dropna(subset=['SKU', 'Toko'])
    top = top.drop_duplicates(subset=['SKU', 'Toko'], keep='first')
    
    print(f"  [DEBUG] _build_stock_top100_map: Built map with {len(top)} SKU+Toko combinations")
    print(f"  [DEBUG] Sample Toko values: {top['Toko'].unique()[:5].tolist()}")
    
    return top


def calculate_inventory_metrics(df_clean, store_location=None):
    """
    Calculate various inventory metrics including safety stock, reorder points, and PO quantities.

    Args:
        df_clean (pd.DataFrame): Input dataframe with required columns
        store_location (str): Store location for applying special SKU rules

    Returns:
        pd.DataFrame: Dataframe with added calculated columns
    """
    import numpy as np
    import pandas as pd

    # Ensure we're working with a copy to avoid SettingWithCopyWarning
    df = df_clean.copy()

    # Set display options
    pd.set_option('display.float_format', '{:.2f}'.format)

    base_stock_col = 'Stok' if 'Stok' in df.columns else 'Stock'
    metric_stock_col = 'stock_top100' if 'stock_top100' in df.columns else base_stock_col

    # Ensure is_top_100_sku exists (0 for non-top-100 by default)
    if 'is_top_100_sku' not in df.columns:
        df['is_top_100_sku'] = 0
    df['is_top_100_sku'] = pd.to_numeric(df['is_top_100_sku'], errors='coerce').fillna(0).astype(int)

    # Force the columns we need into numeric form
    numeric_cols = [
        base_stock_col, metric_stock_col, 'Daily Sales', 'Max. Daily Sales', 'Lead Time',
        'Max. Lead Time', 'Sedang PO', 'HPP', 'Harga', 'sales_contribution'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    try:
        # 1. Safety stock calculation
        df['Safety stock'] = (df['Max. Daily Sales'] * df['Max. Lead Time']) - (df['Daily Sales'] * df['Lead Time'])
        df['Safety stock'] = df['Safety stock'].apply(lambda x: np.ceil(x)).fillna(0).astype(int)

        # 2. Reorder point calculation
        df['Reorder point'] = np.ceil((df['Daily Sales'] * df['Lead Time']) + df['Safety stock']).fillna(0).astype(int)

        # 3. Stock cover - default to 30 days for all SKUs
        df['target_days_cover'] = 30
        
        # Apply special SKU rules if store location provided
        if store_location:
            df = apply_special_sku(df, store_location)

        # Calculate target days cover based on the determined days
        df['qty_for_target_days_cover'] = (
            df['Daily Sales'] * df['target_days_cover']
        ).apply(lambda x: np.ceil(x)).fillna(0).astype(int)

        df['current_days_stock_cover'] = np.where(
            df['Daily Sales'] > 0,
            df[metric_stock_col] / df['Daily Sales'],
            0
        )

        # 5. Is open PO flag
        df['is_open_po'] = np.where(
            (df['current_days_stock_cover'] < df['target_days_cover']) &
            (df[metric_stock_col] <= df['Reorder point']), 1, 0
        )

        # 6. Initial PO quantity
        df['initial_qty_po'] = df['qty_for_target_days_cover'] - df[metric_stock_col] - df.get('Sedang PO', 0)
        df['initial_qty_po'] = (
            pd.Series(
                np.where(df['is_open_po'] == 1, df['initial_qty_po'], 0),
                index=df.index
            )
            .clip(lower=0)
            .astype(int)
        )

        # 7. Emergency PO quantity
        #    For non top-100 SKUs, this must always be 0
        base_emergency = np.where(
            df.get('Sedang PO', 0) > 0,
            np.maximum(0, (df['Max. Lead Time'] - df['current_days_stock_cover']) * df['Daily Sales']),
            np.ceil((df['Max. Lead Time'] - df['current_days_stock_cover']) * df['Daily Sales'])
        )
        df['emergency_po_qty'] = base_emergency * df['is_top_100_sku']
        # Clean up emergency PO quantities
        df['emergency_po_qty'] = (
            df['emergency_po_qty']
            .replace([np.inf, -np.inf], 0)
            .fillna(0)
            .clip(lower=0)
            .astype(int)
        )

        # 8. Updated regular PO quantity:
        # it will be same as initial suggested PO both for SKU Top 100 and others
        df['updated_regular_po_qty'] = df['initial_qty_po'].clip(lower=0).astype(int)

        # 9. Final updated regular PO quantity (enforce minimum order)
        df['final_updated_regular_po_qty'] = np.where(
            (df['updated_regular_po_qty'] > 0) &
            (df['updated_regular_po_qty'] < df['Min. Order']),
            df['Min. Order'],
            df['updated_regular_po_qty']
        ).astype(int)

        # 10. Calculate costs
        df['emergency_po_cost'] = (df['emergency_po_qty'] * df['HPP']).round(2)
        df['final_updated_regular_po_cost'] = (df['final_updated_regular_po_qty'] * df['HPP']).round(2)

        # Clean up any remaining NaN or infinite values
        df = df.fillna(0)

        return df

    except Exception as e:
        print(f"Error in calculate_inventory_metrics: {str(e)}")
        return df_clean


def clean_po_data(df, location=None):
    """Clean and prepare PO data with contribution calculations."""
    try:
        # Create a copy to avoid modifying the original DataFrame
        df = df.copy()
        # remove duplicate SKU
        df = df.drop_duplicates(subset=['SKU'], keep='first')

        # Keep original column names but strip any extra whitespace
        df.columns = df.columns.str.strip()

        # Define required columns (using original case)
        required_columns = [
            'Brand', 'Kategori Brand', 'SKU', 'Nama', 'Toko', 'Stok',
            'Daily Sales', 'Max. Daily Sales', 'Lead Time',
            'Max. Lead Time', 'Min. Order', 'Sedang PO', 'HPP', 'Harga'
        ]

        # Find actual column names in the DataFrame (case-sensitive)
        available_columns = {col.strip(): col for col in df.columns}
        columns_to_keep = []

        for col in required_columns:
            if col in available_columns:
                columns_to_keep.append(available_columns[col])
            else:
                print(f"Warning: Column '{col}' not found in input data")
                # Add as empty column if it's required
                if col in ['Brand', 'SKU', 'HPP', 'Harga']:  # These are critical
                    df[col] = ''

        # Select only the columns we need
        df = df[[col for col in columns_to_keep if col in df.columns]]

        # Check for missing required columns
        missing_columns = [col for col in ['Brand', 'SKU', 'HPP', 'Harga'] if col not in df.columns]
        if missing_columns:
            raise ValueError(
                f"Missing required columns: {missing_columns}. "
                f"Available columns: {df.columns.tolist()}"
            )

        # Clean brand column
        if 'Brand' in df.columns:
            df['Brand'] = df['Brand'].astype(str).str.strip()

        # Convert SKU to string and clean it
        if 'SKU' in df.columns:
            df['SKU'] = df['SKU'].astype(str).str.strip()

        if 'Toko' in df.columns:
            df['Toko'] = df['Toko'].astype(str).str.strip()

        # Convert numeric columns with better error handling
        numeric_columns = [
            'Stok', 'Daily Sales', 'Max. Daily Sales', 'Lead Time',
            'Max. Lead Time', 'Sedang PO', 'HPP', 'Min. Order', 'Harga'
        ]

        for col in numeric_columns:
            if col in df.columns:
                try:
                    # First convert to string, clean, then to numeric
                    df[col] = (
                        df[col]
                        .astype(str)
                        .str.replace(r'[^\d.,-]', '', regex=True)  # Remove non-numeric except .,-
                        .str.replace(',', '.', regex=False)         # Convert commas to decimal points
                        .replace('', '0')                           # Empty strings to '0'
                        .astype(float)                              # Convert to float
                        .fillna(0)                                  # Fill any remaining NaNs with 0
                    )
                except Exception as e:
                    print(f"Warning: Could not convert column '{col}' to numeric: {str(e)}")
                    df[col] = 0  # Set to 0 if conversion fails

        # calculate sales contribution
        df['sales_contribution'] = df['Daily Sales'] * df['Harga']

        return df

    except Exception as e:
        print(f"Error in clean_po_data: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

def _find_sku_col(cols):
    for c in cols:
        if str(c).strip().lower() == "sku":
            return c
    return None

def process_po_file(file_path, supplier_df, running_date=None):
    """Process a single PO file and return merged data and summary."""
    print(f"\nProcessing PO file: {file_path.name} ....")
    
    try:
        # Extract location from filename using the new function
        location = get_store_name_from_filename(file_path.name)
        print(f"  - Extracted location: {location}")
        
        # Read the CSV with error handling
        try:
            is_excel = file_path.suffix.lower() in ['.xlsx', '.xls']

            # Try reading with different encodings if needed
            if is_excel:
                df = read_excel_file(file_path)
            else:
                df = read_csv_file_v2(file_path, decimal_sep='.')
            
            # Check if DataFrame is empty
            if df is None or df.empty:
                raise ValueError("File is empty")
                
            # Clean the data
            df_clean = clean_po_data(df, location)
            # Skip if cleaning failed
            if df_clean is None or df_clean.empty:
                raise ValueError("Data cleaning failed")

            # --- Top 100 SKU handling ---
            df_top_100 = load_top_100_sku_for_store(
                location,
                expected_header_keys=["SKU", "Nama", "Brand", "Toko"]
            )

            # Set is_top_100_sku flag using _find_sku_col for flexible column matching
            top_sku_col = _find_sku_col(df_top_100.columns) if df_top_100 is not None else None
            clean_sku_col = _find_sku_col(df_clean.columns)
            
            print(f"  [DEBUG] top_sku_col: {top_sku_col}, clean_sku_col: {clean_sku_col}")
            
            if (
                df_top_100 is not None
                and not df_top_100.empty
                and top_sku_col is not None
                and clean_sku_col is not None
            ):
                top_skus = set(df_top_100[top_sku_col].astype(str).str.strip())
                print(f"  [DEBUG] Found {len(top_skus)} Top 100 SKUs")
                print(f"  [DEBUG] Sample Top 100 SKUs: {list(top_skus)[:5]}")
                
                df_clean["is_top_100_sku"] = (
                    df_clean[clean_sku_col].astype(str).str.strip().isin(top_skus).astype(int)
                )
                
                top_100_count = df_clean["is_top_100_sku"].sum()
                print(f"  [DEBUG] Marked {top_100_count} rows as Top 100 SKUs in df_clean")
            else:
                print(f"  - No valid Top 100 data for {location}, setting is_top_100_sku = 0")
                df_clean["is_top_100_sku"] = 0
            # --- end Top 100 SKU handling ---

            # stock_top100: use date-specific stock for top-100 rows, else fallback to original stock
            base_stock_col = 'Stok' if 'Stok' in df_clean.columns else 'Stock'
            df_clean['stock_top100'] = pd.to_numeric(df_clean.get(base_stock_col, 0), errors='coerce').fillna(0)
            
            print(f"  [DEBUG] Initialized stock_top100 with base stock column '{base_stock_col}'")
            print(f"  [DEBUG] df_clean has 'Toko' column: {'Toko' in df_clean.columns}")
            if 'Toko' in df_clean.columns:
                print(f"  [DEBUG] Sample Toko values in df_clean: {df_clean['Toko'].unique()[:5].tolist()}")

            top_stock_map = _build_stock_top100_map(df_top_100, running_date)
            if (
                top_stock_map is not None
                and 'SKU' in df_clean.columns
                and 'Toko' in df_clean.columns
            ):
                df_clean['SKU'] = df_clean['SKU'].astype(str).str.strip()
                df_clean['Toko'] = df_clean['Toko'].astype(str).str.strip()

                print(f"  [DEBUG] Merging top_stock_map on ['SKU', 'Toko']")
                
                df_clean = df_clean.merge(
                    top_stock_map,
                    on=['SKU', 'Toko'],
                    how='left',
                    suffixes=('', '_top')
                )

                # Count how many rows got matched
                matched_count = df_clean['stock_top100_top'].notna().sum() if 'stock_top100_top' in df_clean.columns else 0
                print(f"  [DEBUG] Merge matched {matched_count} rows with Top 100 stock data")

                df_clean['stock_top100'] = np.where(
                    df_clean.get('is_top_100_sku', 0).astype(int) == 1,
                    pd.to_numeric(df_clean.get('stock_top100_top', df_clean['stock_top100']), errors='coerce'),
                    df_clean['stock_top100']
                )

                df_clean['stock_top100'] = pd.to_numeric(df_clean['stock_top100'], errors='coerce').fillna(0)

                if 'stock_top100_top' in df_clean.columns:
                    df_clean = df_clean.drop(columns=['stock_top100_top'])
                    
                # Final check
                top_100_with_special_stock = ((df_clean['is_top_100_sku'] == 1) & (df_clean['stock_top100'] != df_clean[base_stock_col])).sum()
                print(f"  [DEBUG] {top_100_with_special_stock} Top 100 SKUs have different stock_top100 values")

            print(f"After excluding specific brands for {location} - current count: ({len(df_clean)})")

            # calculate metrics PO
            df_clean = calculate_inventory_metrics(df_clean, store_location=location)
            
            # Merge with suppliers
            merged_df = merge_with_suppliers(df_clean, supplier_df)
            # Generate summary
            padang_count = (merged_df['Nama Store'] == 'Miss Glam Padang').sum()
            other_supplier_count = ((merged_df['Nama Store'] != 'Miss Glam Padang') & 
                                  (merged_df['Nama Store'] != '')).sum()
            
            summary = {
                'file': file_path.name,
                'location': location,
                'total_rows': len(merged_df),
                'padang_suppliers': int(padang_count),
                'other_suppliers': int(other_supplier_count),
                'no_supplier': int((merged_df['Nama Store'] == '').sum()),
                'status': 'Success'
            }
            
            return merged_df, summary
            
        except Exception as e:
            raise Exception(f"Error processing file data: {str(e)}")
            
    except Exception as e:
        error_msg = f"Error processing {file_path.name}: {str(e)}"
        print(f"  - {error_msg}")
        return None, {
            'file': file_path.name,
            'location': location if 'location' in locals() else 'Unknown',
            'total_rows': 0,
            'padang_suppliers': 0,
            'other_suppliers': 0,
            'no_supplier': 0,
            'status': f"Error: {str(e)[:100]}"
        }

def exclude_specific_brands(df, exclude_brands):
    """Exclude rows where 'Nama' is in the exclude_brands list."""
    return df[~df['Brand'].isin(exclude_brands)]

def clean_and_convert(df):
    """Clean and convert DataFrame columns to appropriate types."""
    if df is None or df.empty:
        return df

    # Make a copy to avoid SettingWithCopyWarning
    df = df.copy()
    
    # Convert all columns to string first to handle NaN/None consistently
    for col in df.columns:
        df[col] = df[col].astype(str)
    
    # Define NA values that should be treated as empty/missing
    na_values = list(NA_VALUES)
    
    # Process each column
    for col in df.columns:
        # Replace NA values with empty string first (treating them as literals, not regex)
        df[col] = df[col].replace(na_values, '', regex=False)
        
        # Skip empty columns
        if df[col].empty:
            continue

        # Convert numeric columns
        if col in NUMERIC_COLUMNS:
            # Convert to numeric, coercing errors to NaN, then fill with 0
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        else:
            # For non-numeric columns, ensure they're strings and strip whitespace
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace('', np.nan).fillna('')
            df[col] = df[col].infer_objects(copy=False)

    return df

def main():
    # Load data
    supplier_df = load_supplier_data(SUPPLIER_PATH)
    all_summaries = []

    current_date = datetime.now().strftime('%Y%m%d')
    run_date = '20251218'
    running_date = datetime.strptime(run_date, '%Y%m%d').strftime('%d %b %Y')
    CURRENT_DIR = INPUT_DIR / run_date

    print(f"Running date for Top 100 stock: {running_date}")

    # Process each PO file
    for file_path in sorted(CURRENT_DIR.glob('*')):
        try:
            merged_df, summary = process_po_file(file_path, supplier_df, running_date=running_date)

            save_to_complete_format(merged_df, file_path.name, file_format='xlsx')
            save_to_complete_format(merged_df, file_path.name)
            save_to_m2_format(merged_df, file_path.name)
            save_to_emergency_po_format(merged_df, file_path.name)

            output_path = OUTPUT_DIR / file_path.name
            summary['output_path'] = str(output_path)

            
            # Print progress
            print(f"  - Location: {summary['location']}")
            print(f"  - Rows processed: {summary['total_rows']}")
            print(f"  - 'Miss Glam Padang' suppliers: {summary['padang_suppliers']} rows")
            print(f"  - Other suppliers: {summary['other_suppliers']} rows")
            print(f"  - No supplier data: {summary['no_supplier']} rows")
            print(f"  - Saved to: {output_path}")
            
            all_summaries.append(summary)
            
        except Exception as e:
            print(f"Error processing {file_path.name}: {str(e)}")
            continue
    
    # Display final summary
    if all_summaries:
        print("\nProcessing complete! Summary:")
        summary_df = pd.DataFrame(all_summaries)
        display(summary_df)
        
        # Show sample of last processed file
        print("\nSample of the last processed file:")
        display(merged_df)
    else:
        print("\nNo files were processed successfully.")

# Run the main function
if __name__ == "__main__":
    main()


Calling _patch_openpyxl_number_casting...
Loading supplier data: /Users/andresuchitra/dev/missglam/autopo/notebook/data/supplier.csv
Running date for Top 100 stock: 18 Dec 2025

Processing PO file: 1. Miss Glam Padang.csv ....
  - Extracted location: PADANG
Loading Top 100 SKU for PADANG from: /Users/andresuchitra/dev/missglam/autopo/notebook/data/top_100_sku/1. Miss Glam Padang.xlsx
Loaded Top 100 SKU for PADANG: 100 rows
  [DEBUG] top_sku_col: SKU, clean_sku_col: SKU
  [DEBUG] Found 100 Top 100 SKUs
  [DEBUG] Sample Top 100 SKUs: ['8998824551285', '8991748011088', '8992222073264', '8993137692335', '792649106211']
  [DEBUG] Marked 100 rows as Top 100 SKUs in df_clean
  [DEBUG] Initialized stock_top100 with base stock column 'Stok'
  [DEBUG] df_clean has 'Toko' column: True
  [DEBUG] Sample Toko values in df_clean: ['Miss Glam Padang']
  [DEBUG] _build_stock_top100_map: Using date column '2025-12-18 00:00:00' for running_date '18 Dec 2025'
  [DEBUG] _build_stock_top100_map: Built map w

/var/folders/8t/7219xcjd2dj829bf02_x9zy80000gn/T/ipykernel_77294/877346394.py:142: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  col_ts = pd.to_datetime(col_str, dayfirst=True, errors='coerce')


File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/complete/1. Miss Glam Padang.xlsx
File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/complete/1. Miss Glam Padang.csv
File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/m2/1. Miss Glam Padang.csv
File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/emergency/1. Miss Glam Padang.csv
  - Location: PADANG
  - Rows processed: 37465
  - 'Miss Glam Padang' suppliers: 11209 rows
  - Other suppliers: 491 rows
  - No supplier data: 25765 rows
  - Saved to: /Users/andresuchitra/dev/missglam/autopo/notebook/output/complete/1. Miss Glam Padang.csv

Processing PO file: 10. Miss Glam Palembang.csv ....
  - Extracted location: PALEMBANG
Loading Top 100 SKU for PALEMBANG from: /Users/andresuchitra/dev/missglam/autopo/notebook/data/top_100_sku/10. Miss Glam Palembang.xlsx
Loaded Top 100 SKU for PALEMBANG: 100 rows
  [DEBUG] top_sku_col: SKU, clean_sku_col: SKU
  [DEBUG] Found

/var/folders/8t/7219xcjd2dj829bf02_x9zy80000gn/T/ipykernel_77294/877346394.py:142: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  col_ts = pd.to_datetime(col_str, dayfirst=True, errors='coerce')


File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/complete/10. Miss Glam Palembang.xlsx
File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/complete/10. Miss Glam Palembang.csv
File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/m2/10. Miss Glam Palembang.csv
File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/emergency/10. Miss Glam Palembang.csv
  - Location: PALEMBANG
  - Rows processed: 37064
  - 'Miss Glam Padang' suppliers: 209 rows
  - Other suppliers: 11307 rows
  - No supplier data: 25548 rows
  - Saved to: /Users/andresuchitra/dev/missglam/autopo/notebook/output/complete/10. Miss Glam Palembang.csv

Processing PO file: 11. Miss Glam Damar.csv ....
  - Extracted location: DAMAR
Loading Top 100 SKU for DAMAR from: /Users/andresuchitra/dev/missglam/autopo/notebook/data/top_100_sku/11. Miss Glam Damar.xlsx
Loaded Top 100 SKU for DAMAR: 100 rows
  [DEBUG] top_sku_col: SKU, clean_sku_col: SKU
  [DEBUG] Fo

...

/var/folders/8t/7219xcjd2dj829bf02_x9zy80000gn/T/ipykernel_77294/877346394.py:142: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  col_ts = pd.to_datetime(col_str, dayfirst=True, errors='coerce')


File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/complete/9. Miss Glam Medan.xlsx
File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/complete/9. Miss Glam Medan.csv
File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/m2/9. Miss Glam Medan.csv
File saved to /Users/andresuchitra/dev/missglam/autopo/notebook/output/emergency/9. Miss Glam Medan.csv
  - Location: MEDAN
  - Rows processed: 37193
  - 'Miss Glam Padang' suppliers: 189 rows
  - Other suppliers: 11389 rows
  - No supplier data: 25615 rows
  - Saved to: /Users/andresuchitra/dev/missglam/autopo/notebook/output/complete/9. Miss Glam Medan.csv

Processing complete! Summary:


,file,location,total_rows,padang_suppliers,other_suppliers,no_supplier,status,output_path
0,1. Miss Glam Padang.csv,PADANG,37465,11209,491,25765,Success,/Users/andresuchitra/dev/missglam/autopo/noteb...
1,10. Miss Glam Palembang.csv,PALEMBANG,37064,209,11307,25548,Success,/Users/andresuchitra/dev/missglam/autopo/noteb...
2,11. Miss Glam Damar.csv,DAMAR,37537,6,11689,25842,Success,/Users/andresuchitra/dev/missglam/autopo/noteb...
3,12. Miss Glam Bangka.csv,BANGKA,37028,274,11209,25545,Success,/Users/andresuchitra/dev/missglam/autopo/noteb...
4,13. Miss Glam Payakumbuh.csv,PAYAKUMBUH,37117,112,11397,25608,Success,/Users/andresuchitra/dev/missglam/autopo/noteb...
5,14. Miss Glam Solok.csv,SOLOK,37108,110,11398,25600,Success,/Users/andresuchitra/dev/missglam/autopo/noteb...
6,15. Miss Glam Tembilahan.csv,TEMBILAHAN,36954,202,11207,25545,Success,/Users/andresuchitra/dev/missglam/autopo/noteb...
7,16. Miss Glam Lubuk Linggau.csv,LUBUK LINGGAU,37004,275,11153,25576,Success,/Users/andresuchitra/dev/missglam/autopo/noteb...
8,17. Miss Glam Dumai.csv,DUMAI,37097,170,11344,25583,Success,/Users/andresuchitra/dev/missglam/autopo/noteb...
9,18. Miss Glam Kedaton.csv,KEDATON,37054,200,11256,25598,Success,/Users/andresuchitra/dev/missglam/autopo/noteb...



Sample of the last processed file:


,Brand,Kategori Brand,SKU,Nama,Toko,Stok,Daily Sales,Max. Daily Sales,Lead Time,Max. Lead Time,...,Nama Supplier,ID Brand,Nama Brand,ID Store,Nama Store,Hari Order,Min. Purchase,Trading Term,Promo Factor,Delay Factor
0,ACNAWAY,BRAND VIRAL,10400614911,ACNAWAY 3 in 1 Acne Sun Serum Sunscreen Serum ...,Miss Glam Medan,6.00,0.03,1.00,6.00,28.00,...,PT. BERSAMA DISTRIVERSA INDONESIA (DC CIPUTAT),1480.00,ACNAWAY,19.00,Miss Glam Medan,2.00,500000.00,0.00,,
1,ACNAWAY,BRAND VIRAL,10500210743,ACNAWAY Advanced Mugwort Gel Mask BIG SIZE 100ml,Miss Glam Medan,22.00,0.03,0.00,6.00,28.00,...,PT. BERSAMA DISTRIVERSA INDONESIA (DC CIPUTAT),1480.00,ACNAWAY,19.00,Miss Glam Medan,2.00,500000.00,0.00,,
2,ACNAWAY,BRAND VIRAL,10100824612,ACNAWAY Mugwort Acne Clear Bar Soap 100gr,Miss Glam Medan,20.00,0.07,0.00,6.00,28.00,...,PT. BERSAMA DISTRIVERSA INDONESIA (DC CIPUTAT),1480.00,ACNAWAY,19.00,Miss Glam Medan,2.00,500000.00,0.00,,
3,ACNAWAY,BRAND VIRAL,11200219943,ACNAWAY Mugwort Blackhead Treatment Step 2 17ml,Miss Glam Medan,18.00,0.00,0.00,0.00,0.00,...,PT. BERSAMA DISTRIVERSA INDONESIA (DC CIPUTAT),1480.00,ACNAWAY,19.00,Miss Glam Medan,2.00,500000.00,0.00,,
4,ACNAWAY,BRAND VIRAL,10400517459,ACNAWAY Mugwort Daily Sunscreen Only For Acne ...,Miss Glam Medan,3.00,0.42,3.00,6.00,28.00,...,PT. BERSAMA DISTRIVERSA INDONESIA (DC CIPUTAT),1480.00,ACNAWAY,19.00,Miss Glam Medan,2.00,500000.00,0.00,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37188,ZYWELL,DELISTING,10500322936,ZYWELL Pell Off Mask Cellendula 10g,Miss Glam Medan,0.00,0.00,0.00,0.00,0.00,...,,0.00,NaN,0.00,,0.00,0.00,0.00,,
37189,ZYWELL,DELISTING,18200200793,ZYWELL Pell Off Mask CHamomille 10g,Miss Glam Medan,0.00,0.00,0.00,0.00,0.00,...,,0.00,NaN,0.00,,0.00,0.00,0.00,,
37190,ZYWELL,DELISTING,10500300101,ZYWELL Pell Off Mask Jasmine 10g,Miss Glam Medan,0.00,0.00,0.00,0.00,0.00,...,,0.00,NaN,0.00,,0.00,0.00,0.00,,
37191,ZYWELL,DELISTING,10500322858,ZYWELL Pell Off Mask Rose 10g,Miss Glam Medan,0.00,0.00,0.00,0.00,0.00,...,,0.00,NaN,0.00,,0.00,0.00,0.00,,


# 18 Dec
1. kolom stok utk kalkulasi suggested PO, utk top 100 SKU memakai data toko `top_100_sku` - column 18 Dec 2025
2. `target_days_cover` memakai excel baru utk beberapa SKU dan store -> file terpisah